# UnifyWeaver में उन्नत रिकर्सन पैटर्न

यह नोटबुक चार मुख्य रिकर्सन पैटर्न प्रदर्शित करती है जिन्हें UnifyWeaver पहचान सकता है और अनुकूलित कर सकता है:

1. **टेल रिकर्सन (Tail Recursion)** - संचायक (accumulator) के साथ पुनरावृत्त लूप
2. **रैखिक रिकर्सन (Linear Recursion)** - मेमोइज़ेशन के साथ एकल पुनरावर्ती कॉल
3. **ट्री रिकर्सन (Tree Recursion)** - संरचना भागों पर कई पुनरावर्ती कॉल
4. **म्यूचुअल रिकर्सन (Mutual Recursion)** - चक्रों में एक दूसरे को कॉल करने वाले प्रेडिकेट्स

## सीखने के उद्देश्य

- विभिन्न रिकर्सन पैटर्न को समझना
- देखना कि UnifyWeaver प्रत्येक पैटर्न का पता कैसे लगाता है और उसे कैसे अनुकूलित करता है
- प्रदर्शन विशेषताओं की तुलना करना
- प्रत्येक पैटर्न का उपयोग कब करना है यह सीखना

## सेटअप

UnifyWeaver वातावरण प्रारंभ करें।

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## पैटर्न 1: टेल रिकर्सन (Tail Recursion)

टेल रिकर्सन मध्यवर्ती परिणामों को आगे बढ़ाने के लिए एक संचायक का उपयोग करता है, और पुनरावर्ती कॉल फ़ंक्शन में **अंतिम क्रिया** होती है।

### उदाहरण: सूची में तत्वों की गिनती

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### Prolog में परीक्षण करें

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### पैटर्न पहचान की जांच करें

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Bash में संकलित करें

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### जनरेट किए गए Bash का परीक्षण करें

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## पैटर्न 2: रैखिक रिकर्सन (Linear Recursion)

रैखिक रिकर्सन में प्रति क्लॉज **सटीक रूप से एक** पुनरावर्ती कॉल होती है, जिसमें पुनरावर्ती कॉल वापस आने के बाद गणना होती है।

### उदाहरण: फैक्टोरियल (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### Prolog में परीक्षण करें

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### पैटर्न पहचान की जांच करें

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Bash में संकलित करें

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### जनरेट किए गए Bash का परीक्षण करें

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## पैटर्न 3: ट्री रिकर्सन (Tree Recursion)

संरचना के विभिन्न भागों को संसाधित करने के लिए ट्री रिकर्सन **एकाधिक** पुनरावर्ती कॉल करता है।

### उदाहरण: ट्री सम (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### Prolog में परीक्षण करें

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Bash में संकलित करें

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### जनरेट किए गए Bash का परीक्षण करें

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## पैटर्न 4: म्यूचुअल रिकर्सन (Mutual Recursion)

म्यूचुअल रिकर्सन तब होता है जब दो या दो से अधिक प्रेडिकेट्स एक चक्र में एक दूसरे को कॉल करते हैं।

### उदाहरण: सम (Even) और विषम (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### Prolog में परीक्षण करें

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### म्यूचुअल रिकर्सन की जांच करें

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Bash में संकलित करें

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### जनरेट किए गए Bash का परीक्षण करें

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## पैटर्न तुलना

आइए प्रत्येक पैटर्न की विशेषताओं की तुलना करें:

| पैटर्न | पुनरावर्ती कॉल | अनुकूलन | अंतरिक्ष जटिलता | इसके लिए सर्वोत्तम |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **टेल** | 1 (टेल स्थिति में) | पुनरावृत्त लूप | O(1) | संचायक, रैखिक स्कैन |
| **रैखिक** | 1 (किसी भी स्थिति में) | फोल्ड + मेमोइज़ेशन | O(n) मेमो तालिका | फाइबोनैचि, फैक्टोरियल |
| **ट्री** | 2+ (संरचना भाग) | संरचनात्मक अपघटन | O(गहराई) स्टैक | ट्री/ग्राफ संचालन |
| **म्यूचुअल** | 1+ (प्रेडिकेट्स के पार) | साझा मेमोइज़ेशन | O(n) साझा तालिका | सम/विषम, पारस्परिक परिभाषाएं |

## पैटर्न पहचान क्रम

UnifyWeaver इस क्रम में पैटर्न से मिलान करने का प्रयास करता है:

1. **टेल रिकर्सन** (सबसे कुशल)
2. **रैखिक रिकर्सन** (जब तक निषिद्ध न हो)
3. **ट्री रिकर्सन** (संरचनात्मक)
4. **म्यूचुअल रिकर्सन** (SCC पहचान)
5. **मूल रिकर्सन** (डिफ़ॉल्ट फॉलबैक)

आप `forbid_linear_recursion/1` के साथ पहचान को प्रभावित कर सकते हैं।

## अभ्यास: आपकी बारी!

इन प्रेडिकेट्स को परिभाषित और संकलित करने का प्रयास करें:

### 1. टेल रिकर्सिव सम
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. रैखिक रिकर्सिव फाइबोनैचि
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. ट्री की ऊंचाई
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## सारांश

इस नोटबुक में, आपने सीखा:

✅ UnifyWeaver में चार मुख्य रिकर्सन पैटर्न

✅ Prolog में प्रत्येक पैटर्न को कैसे परिभाषित करें

✅ UnifyWeaver प्रत्येक पैटर्न का पता कैसे लगाता है और उसे कैसे अनुकूलित करता है

✅ प्रत्येक पैटर्न की प्रदर्शन विशेषताएं

✅ प्रत्येक पैटर्न का उपयोग कब करना है

## अगले कदम

उन्नत कोड विश्लेषण और विज़ुअलाइज़ेशन के बारे में जानने के लिए **नोटबुक 3: कॉल ग्राफ विज़ुअलाइज़ेशन** पर आगे बढ़ें!